# PREPROCESSING DATA NTHUD

# Library Yang Dibutuhkan

In [1]:

# Import library dasar untuk manipulasi file dan data
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE" # untuk menghindari error pada beberapa sistem dan memakai dua dua versi library OpenMP 
import shutil #untuk mengatur file dan folder (seperti menghapus, memindah, menyalin, dll)
from pathlib import Path

# Import library untuk numerik dan visualisasi
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import library untuk image processing
from PIL import Image
import cv2

# Import library PyTorch (untuk cek apakah environment sudah siap)
import torch
import torchvision

# Konfigurasi visual
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print(f" PyTorch Version: {torch.__version__}")
print(f" Torchvision Version: {torchvision.__version__}")
print(f" CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU Device: {torch.cuda.get_device_name(0)}")


 PyTorch Version: 2.7.1+cu118
 Torchvision Version: 0.22.1+cu118
 CUDA Available: True
   GPU Device: NVIDIA GeForce RTX 3060


# Pembagian data train, val, test dengan rasio 61:28:10

In [2]:
import os
import shutil
from collections import Counter

def parse_subject_id(filename):
    """
    Ekstrak ID subjek dari awal nama file NTHUDDD2.
    Contoh: "001_glasses_slowBlinkWithNodding_1919_drowsy.jpg" -> "001"
    """
    base = os.path.splitext(os.path.basename(filename))[0]
    parts = base.split("_")
    return parts[0] if len(parts) > 0 else base

def split_dataset_nthu_final(source_dir, dest_dir):
    # ===========================
    # PEMBAGIAN ID SUBJEK (Diperbarui sesuai koreksi Daffa)
    # ===========================
    VAL_IDS = ['002']
    TEST_IDS = ['006']
    # Subjek '005' dan '001' otomatis akan masuk ke TRAIN
    
    # 1. Cek ketersediaan kelas (Seharusnya "drowsy" dan "notdrowsy")
    classes = [d for d in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, d))]
    classes = sorted(classes) 
    
    if not classes:
        print(" [!] Tidak menemukan folder kelas. Pastikan path SOURCE benar.")
        return

    # 2. Buat struktur folder tujuan
    splits_list = ["train", "val", "test"]
    for split in splits_list:
        for cls in classes:
            os.makedirs(os.path.join(dest_dir, split, cls), exist_ok=True)

    print(f"Mulai memproses dataset dari: {source_dir}")
    print(f"Hasil akan disimpan di      : {dest_dir}\n")

    # 3. Iterasi file dan copy berdasarkan ID Subjek
    counts = Counter()

    for cls in classes:
        cls_dir = os.path.join(source_dir, cls)
        files = [f for f in os.listdir(cls_dir) if f.lower().endswith((".jpg",".jpeg",".png"))]
        
        print(f"Memproses kelas {cls} ({len(files)} gambar)...")
        
        for f in files:
            subj_id = parse_subject_id(f)
            
            # Tentukan split berdasarkan ID subjek
            if subj_id in TEST_IDS:
                target_split = "test"
            elif subj_id in VAL_IDS:
                target_split = "val"
            else:
                target_split = "train"
            
            src_path = os.path.join(cls_dir, f)
            dst_path = os.path.join(dest_dir, target_split, cls, f)
            
            # Copy file ke folder baru
            shutil.copy2(src_path, dst_path)
            counts[target_split] += 1

    # 4. Laporan Hasil
    total_all = sum(counts.values())
    print("\n" + "="*55)
    print(f"{'LAPORAN AKHIR SPLIT DATASET NTHUDDD2':^55}")
    print("="*55)
    for s in splits_list:
        perc = (counts[s]/total_all)*100 if total_all > 0 else 0
        print(f" {s.upper():5} SET : {counts[s]:6} gambar ({perc:.2f}%)")
    print("-" * 55)
    print(f" TOTAL DATA : {total_all} gambar")
    print("="*55)

def verify_nthu_leakage(dest_dir):
    splits = ["train", "val", "test"]
    split_groups = {}

    for s in splits:
        ids = set()
        split_path = os.path.join(dest_dir, s)
        for root, dirs, files in os.walk(split_path):
            for f in files:
                if f.lower().endswith((".jpg", ".png", ".jpeg")):
                    ids.add(parse_subject_id(f))
        split_groups[s] = ids

    print("\n=== VERIFIKASI SUBJEK (ANTI LEAKAGE) ===")
    print(f"Subjek di Train : {split_groups['train']} (Total: {len(split_groups['train'])})")
    print(f"Subjek di Val   : {split_groups['val']} (Total: {len(split_groups['val'])})")
    print(f"Subjek di Test  : {split_groups['test']} (Total: {len(split_groups['test'])})")

    # Cek irisan (Intersection) untuk memastikan tidak ada subjek yang bocor
    leak_tv = split_groups["train"] & split_groups["val"]
    leak_tt = split_groups["train"] & split_groups["test"]
    leak_vt = split_groups["val"] & split_groups["test"]
    
    if not leak_tv and not leak_tt and not leak_vt:
        print("\n[STATUS] AMAN 100%! Tidak ada subjek yang bocor antar folder.")
    else:
        print("\n[WARNING] Terjadi kebocoran subjek! Cek kembali penamaan file.")

# =====================================================================
# EKSEKUSI
# =====================================================================
# Path asli NTHUDDD2 (Hanya berisi folder "drowsy" dan "notdrowsy")
source_dataset = r"C:\kuliah-sementara\SKRIPSI\Dataset_nthuddd2" 

# Folder baru yang akan berisi train, val, dan test
dest_dataset = r"C:\kuliah-sementara\SKRIPSI\Dataset_nthuddd2_SPLIT"

if os.path.exists(dest_dataset):
    print(f"Menghapus folder lama: {dest_dataset}")
    shutil.rmtree(dest_dataset)

# Jalankan proses split
split_dataset_nthu_final(source_dataset, dest_dataset)

# Verifikasi kebocoran data
verify_nthu_leakage(dest_dataset)


Mulai memproses dataset dari: C:\kuliah-sementara\SKRIPSI\Dataset_nthuddd2
Hasil akan disimpan di      : C:\kuliah-sementara\SKRIPSI\Dataset_nthuddd2_SPLIT

Memproses kelas drowsy (36034 gambar)...
Memproses kelas notdrowsy (30491 gambar)...

         LAPORAN AKHIR SPLIT DATASET NTHUDDD2          
 TRAIN SET :  40949 gambar (61.55%)
 VAL   SET :  18833 gambar (28.31%)
 TEST  SET :   6743 gambar (10.14%)
-------------------------------------------------------
 TOTAL DATA : 66525 gambar

=== VERIFIKASI SUBJEK (ANTI LEAKAGE) ===
Subjek di Train : {'001', '005'} (Total: 2)
Subjek di Val   : {'002'} (Total: 1)
Subjek di Test  : {'006'} (Total: 1)

[STATUS] AMAN 100%! Tidak ada subjek yang bocor antar folder.


# Konfigurasi Path Dataset

In [3]:

# ===========================
# UBAH PATH INI SESUAI LOKASI DATASET ANDA
# ===========================

# Path utama dataset (parent folder) yang sudah di-split
DATASET_ROOT = r"C:\kuliah-sementara\SKRIPSI\Dataset_nthuddd2_SPLIT" 

# Path untuk setiap split
TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
VAL_DIR = os.path.join(DATASET_ROOT, "val")
TEST_DIR = os.path.join(DATASET_ROOT, "test")

# Nama kelas (folder di dalam train/val/test)
CLASS_NAMES = ["notdrowsy", "drowsy"]  

# Cek apakah folder-folder ini ada
print(" Memeriksa struktur folder dataset...\n")

for split_name, split_path in [("Train", TRAIN_DIR), ("Val", VAL_DIR), ("Test", TEST_DIR)]:
    if os.path.exists(split_path):
        print(f" {split_name} folder ditemukan: {split_path}")
        for class_name in CLASS_NAMES:
            class_path = os.path.join(split_path, class_name)
            if os.path.exists(class_path):
                print(f"   ├─  Kelas '{class_name}' ditemukan")
            else:
                print(f"   ├─  Kelas '{class_name}' TIDAK DITEMUKAN!")
    else:
        print(f" {split_name} folder TIDAK DITEMUKAN: {split_path}")

print("\n" + "="*60)


 Memeriksa struktur folder dataset...

 Train folder ditemukan: C:\kuliah-sementara\SKRIPSI\Dataset_nthuddd2_SPLIT\train
   ├─  Kelas 'notdrowsy' ditemukan
   ├─  Kelas 'drowsy' ditemukan
 Val folder ditemukan: C:\kuliah-sementara\SKRIPSI\Dataset_nthuddd2_SPLIT\val
   ├─  Kelas 'notdrowsy' ditemukan
   ├─  Kelas 'drowsy' ditemukan
 Test folder ditemukan: C:\kuliah-sementara\SKRIPSI\Dataset_nthuddd2_SPLIT\test
   ├─  Kelas 'notdrowsy' ditemukan
   ├─  Kelas 'drowsy' ditemukan



# Fungsi untuk Menghitung Jumlah File per Kelas

In [4]:

def count_images_in_folder(folder_path):
    """
    Menghitung jumlah file gambar (.jpg, .png, .jpeg) di dalam folder.
    
    Args:
        folder_path (str): Path ke folder yang akan dihitung.
        
    Returns:
        int: Jumlah file gambar.
    """
    if not os.path.exists(folder_path):
        return 0
    
    valid_extensions = ['.jpg', '.jpeg', '.png', '.bmp']
    count = 0
    
    for file in os.listdir(folder_path):
        if any(file.lower().endswith(ext) for ext in valid_extensions):
            count += 1
    
    return count


def get_dataset_statistics(base_dir, class_names):
    """
    Menghitung statistik dataset untuk setiap split dan kelas.
    
    Args:
        base_dir (str): Path ke folder split (train/val/test).
        class_names (list): Daftar nama kelas (misalnya ['wake', 'sleep']).
        
    Returns:
        dict: Dictionary berisi jumlah gambar per kelas.
    """
    stats = {}
    
    for class_name in class_names:
        class_path = os.path.join(base_dir, class_name)
        stats[class_name] = count_images_in_folder(class_path)
    
    return stats

# Menghitung dan Menampilkan Statistik Dataset

In [5]:

# Hitung jumlah gambar di setiap split
print(" STATISTIK DATASET\n")
print("="*60)

splits = {
    "Train": TRAIN_DIR,
    "Val": VAL_DIR,
    "Test": TEST_DIR
}

all_stats = {}

for split_name, split_path in splits.items():
    stats = get_dataset_statistics(split_path, CLASS_NAMES)
    all_stats[split_name] = stats
    
    total = sum(stats.values())
    
    print(f"\n {split_name.upper()} SET:")
    print(f"   ├─ notdrowsy: {stats['notdrowsy']:,} gambar")
    print(f"   ├─ drowsy: {stats['drowsy']:,} gambar")
    print(f"   └─ Total: {total:,} gambar")
    
    # Hitung balance ratio
    if total > 0:
        notdrowsy_ratio = (stats['notdrowsy'] / total) * 100
        drowsy_ratio = (stats['drowsy'] / total) * 100
        print(f"      Balance: {notdrowsy_ratio:.2f}% notdrowsy | {drowsy_ratio:.2f}% drowsy")

print("\n" + "="*60)

# Ringkasan total keseluruhan
total_all = sum([sum(stats.values()) for stats in all_stats.values()])
print(f"\n TOTAL DATASET: {total_all:,} gambar")

 STATISTIK DATASET


 TRAIN SET:
   ├─ notdrowsy: 18,278 gambar
   ├─ drowsy: 22,671 gambar
   └─ Total: 40,949 gambar
      Balance: 44.64% notdrowsy | 55.36% drowsy

 VAL SET:
   ├─ notdrowsy: 8,237 gambar
   ├─ drowsy: 10,596 gambar
   └─ Total: 18,833 gambar
      Balance: 43.74% notdrowsy | 56.26% drowsy

 TEST SET:
   ├─ notdrowsy: 3,976 gambar
   ├─ drowsy: 2,767 gambar
   └─ Total: 6,743 gambar
      Balance: 58.96% notdrowsy | 41.04% drowsy


 TOTAL DATASET: 66,525 gambar


# Fungsi Cek dan Hapus File Corrupt

In [6]:

def check_and_remove_corrupt_images(base_dir, class_names):
    """
    Memeriksa semua gambar di dalam folder dan menghapus file yang corrupt.
    
    Args:
        base_dir (str): Path ke folder split (train/val/test).
        class_names (list): Daftar nama kelas.
        
    Returns:
        list: Daftar file yang dihapus karena corrupt.
    """
    corrupt_files = []
    
    for class_name in class_names:
        class_path = os.path.join(base_dir, class_name)
        
        if not os.path.exists(class_path):
            print(f"  Folder tidak ditemukan: {class_path}")
            continue
        
        print(f"\n Memeriksa folder: {class_path}")
        
        for filename in os.listdir(class_path):
            file_path = os.path.join(class_path, filename)
            
            # Skip jika bukan file
            if not os.path.isfile(file_path):
                continue
            
            try:
                # Coba buka gambar menggunakan PIL
                img = Image.open(file_path)
                img.verify()  # Verifikasi integritas file
                
                # Coba load sebagai array (double check)
                img = Image.open(file_path)
                img.load()
                
            except Exception as e:
                print(f"    File corrupt: {filename} - Error: {str(e)}")
                corrupt_files.append(file_path)
                
                # Hapus file corrupt
                try:
                    os.remove(file_path)
                    print(f"        File telah dihapus.")
                except Exception as del_error:
                    print(f"        Gagal menghapus: {str(del_error)}")
    
    return corrupt_files

# Jalankan Pengecekan File Corrupt

In [7]:

print(" MEMULAI PENGECEKAN FILE CORRUPT...\n")
print("="*60)

all_corrupt_files = []

for split_name, split_path in splits.items():
    print(f"\n{'='*60}")
    print(f" SPLIT: {split_name.upper()}")
    print(f"{'='*60}")
    
    corrupt_files = check_and_remove_corrupt_images(split_path, CLASS_NAMES)
    all_corrupt_files.extend(corrupt_files)

print("\n" + "="*60)
print(f"\n HASIL PENGECEKAN:")
print(f"   Total file corrupt yang ditemukan dan dihapus: {len(all_corrupt_files)}")

if len(all_corrupt_files) == 0:
    print("    SEMUA FILE DALAM KONDISI BAIK!")
else:
    print("\n   Daftar file yang dihapus:")
    for file in all_corrupt_files:
        print(f"   - {file}")

print("="*60)


 MEMULAI PENGECEKAN FILE CORRUPT...


 SPLIT: TRAIN

 Memeriksa folder: C:\kuliah-sementara\SKRIPSI\Dataset_nthuddd2_SPLIT\train\notdrowsy

 Memeriksa folder: C:\kuliah-sementara\SKRIPSI\Dataset_nthuddd2_SPLIT\train\drowsy

 SPLIT: VAL

 Memeriksa folder: C:\kuliah-sementara\SKRIPSI\Dataset_nthuddd2_SPLIT\val\notdrowsy

 Memeriksa folder: C:\kuliah-sementara\SKRIPSI\Dataset_nthuddd2_SPLIT\val\drowsy

 SPLIT: TEST

 Memeriksa folder: C:\kuliah-sementara\SKRIPSI\Dataset_nthuddd2_SPLIT\test\notdrowsy

 Memeriksa folder: C:\kuliah-sementara\SKRIPSI\Dataset_nthuddd2_SPLIT\test\drowsy


 HASIL PENGECEKAN:
   Total file corrupt yang ditemukan dan dihapus: 0
    SEMUA FILE DALAM KONDISI BAIK!


# Fungsi Cek dan Hapus File Corrupt

In [8]:

def check_and_remove_corrupt_images(base_dir, class_names):
    """
    Memeriksa semua gambar di dalam folder dan menghapus file yang corrupt.
    
    Args:
        base_dir (str): Path ke folder split (train/val/test).
        class_names (list): Daftar nama kelas.
        
    Returns:
        list: Daftar file yang dihapus karena corrupt.
    """
    corrupt_files = []
    
    for class_name in class_names:
        class_path = os.path.join(base_dir, class_name)
        
        if not os.path.exists(class_path):
            print(f"  Folder tidak ditemukan: {class_path}")
            continue
        
        print(f"\n Memeriksa folder: {class_path}")
        
        for filename in os.listdir(class_path):
            file_path = os.path.join(class_path, filename)
            
            # Skip jika bukan file
            if not os.path.isfile(file_path):
                continue
            
            try:
                # Coba buka gambar menggunakan PIL
                img = Image.open(file_path)
                img.verify()  # Verifikasi integritas file
                
                # Coba load sebagai array (double check)
                img = Image.open(file_path)
                img.load()
                
            except Exception as e:
                print(f"    File corrupt: {filename} - Error: {str(e)}")
                corrupt_files.append(file_path)
                
                # Hapus file corrupt
                try:
                    os.remove(file_path)
                    print(f"        File telah dihapus.")
                except Exception as del_error:
                    print(f"        Gagal menghapus: {str(del_error)}")
    
    return corrupt_files

# Jalankan Pengecekan File Corrupt

In [9]:

print(" MEMULAI PENGECEKAN FILE CORRUPT...\n")
print("="*60)

all_corrupt_files = []

for split_name, split_path in splits.items():
    print(f"\n{'='*60}")
    print(f" SPLIT: {split_name.upper()}")
    print(f"{'='*60}")
    
    corrupt_files = check_and_remove_corrupt_images(split_path, CLASS_NAMES)
    all_corrupt_files.extend(corrupt_files)

print("\n" + "="*60)
print(f"\n HASIL PENGECEKAN:")
print(f"   Total file corrupt yang ditemukan dan dihapus: {len(all_corrupt_files)}")

if len(all_corrupt_files) == 0:
    print("    SEMUA FILE DALAM KONDISI BAIK!")
else:
    print("\n   Daftar file yang dihapus:")
    for file in all_corrupt_files:
        print(f"   - {file}")

print("="*60)


 MEMULAI PENGECEKAN FILE CORRUPT...


 SPLIT: TRAIN

 Memeriksa folder: C:\kuliah-sementara\SKRIPSI\Dataset_nthuddd2_SPLIT\train\notdrowsy

 Memeriksa folder: C:\kuliah-sementara\SKRIPSI\Dataset_nthuddd2_SPLIT\train\drowsy

 SPLIT: VAL

 Memeriksa folder: C:\kuliah-sementara\SKRIPSI\Dataset_nthuddd2_SPLIT\val\notdrowsy

 Memeriksa folder: C:\kuliah-sementara\SKRIPSI\Dataset_nthuddd2_SPLIT\val\drowsy

 SPLIT: TEST

 Memeriksa folder: C:\kuliah-sementara\SKRIPSI\Dataset_nthuddd2_SPLIT\test\notdrowsy

 Memeriksa folder: C:\kuliah-sementara\SKRIPSI\Dataset_nthuddd2_SPLIT\test\drowsy


 HASIL PENGECEKAN:
   Total file corrupt yang ditemukan dan dihapus: 0
    SEMUA FILE DALAM KONDISI BAIK!


# Visualisasi Sampel Gambar dari Dataset

In [10]:

def calculate_dataset_stats(base_dir, class_names):
    """
    Menghitung jumlah gambar dalam setiap kelas di sebuah direktori.
    """
    stats = {}
    for class_name in class_names:
        class_path = os.path.join(base_dir, class_name)
        if os.path.exists(class_path):
            files = [f for f in os.listdir(class_path) if os.path.isfile(os.path.join(class_path, f))]
            stats[class_name] = len(files)
        else:
            stats[class_name] = 0
            
    return stats

print(" DISTRIBUSI KELAS DATASET\n")
print("="*60)

total_all = 0
split_stats = {}

for split_name, split_path in splits.items():
    stats = calculate_dataset_stats(split_path, CLASS_NAMES)
    split_stats[split_name] = stats
    
    total = sum(stats.values())
    total_all += total
    
    print(f"\n SPLIT: {split_name.upper()}")
    print("-" * 30)
    print(f"   ├─ notdrowsy: {stats.get('notdrowsy', 0):,} gambar")
    print(f"   ├─ drowsy: {stats.get('drowsy', 0):,} gambar")
    print(f"   └─ Total {split_name}: {total:,} gambar")
    
    if total > 0:
        notdrowsy_ratio = (stats.get('notdrowsy', 0) / total) * 100
        drowsy_ratio = (stats.get('drowsy', 0) / total) * 100
        print(f"      Balance: {notdrowsy_ratio:.2f}% notdrowsy | {drowsy_ratio:.2f}% drowsy")

print("\n" + "="*60)
print(f" TOTAL KESELURUHAN DATASET: {total_all:,} gambar")
print("="*60)


 DISTRIBUSI KELAS DATASET


 SPLIT: TRAIN
------------------------------
   ├─ notdrowsy: 18,278 gambar
   ├─ drowsy: 22,671 gambar
   └─ Total Train: 40,949 gambar
      Balance: 44.64% notdrowsy | 55.36% drowsy

 SPLIT: VAL
------------------------------
   ├─ notdrowsy: 8,237 gambar
   ├─ drowsy: 10,596 gambar
   └─ Total Val: 18,833 gambar
      Balance: 43.74% notdrowsy | 56.26% drowsy

 SPLIT: TEST
------------------------------
   ├─ notdrowsy: 3,976 gambar
   ├─ drowsy: 2,767 gambar
   └─ Total Test: 6,743 gambar
      Balance: 58.96% notdrowsy | 41.04% drowsy

 TOTAL KESELURUHAN DATASET: 66,525 gambar


# Chek Duplikasi

In [11]:


import hashlib
import os

def calculate_md5(file_path):
    """Menghitung MD5 hash dari sebuah file gambar."""
    hash_md5 = hashlib.md5()
    try:
        with open(file_path, "rb") as f:
            for chunk in iter(lambda: f.read(4096), b""):
                hash_md5.update(chunk)
        return hash_md5.hexdigest()
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return None

def find_and_remove_duplicates(train_dir, val_dir, test_dir):
    """
    Mencari dan MENGHAPUS gambar duplikat antar folder split.
    Prioritas Hapus: Hapus file di Test/Val, pertahankan file di Train.
    """
    print(" SEDANG MEMERIKSA & MEMBERSIHKAN DUPLIKASI KONTEN...")
    print("   (Proses ini mungkin memakan waktu 1-2 menit...)")
    
    train_hashes = {}
    val_hashes = {}
    test_hashes = {}
    
    def fill_hashes(directory, hash_dict, label):
        print(f"    Memproses {label} set...")
        count = 0
        for root, _, files in os.walk(directory):
            for file in files:
                if file.lower().endswith(('.jpg', '.png', '.jpeg')):
                    path = os.path.join(root, file)
                    file_hash = calculate_md5(path)
                    if file_hash:
                        hash_dict[file_hash] = path
                        count += 1
        return count

    # 1. Hitung Hash
    fill_hashes(train_dir, train_hashes, "TRAIN")
    fill_hashes(val_dir, val_hashes, "VAL")
    fill_hashes(test_dir, test_hashes, "TEST")
    
    # 2. Cek Irisan & Hapus
    train_keys = set(train_hashes.keys())
    val_keys = set(val_hashes.keys())
    test_keys = set(test_hashes.keys())
    
    leak_train_test = train_keys.intersection(test_keys)
    leak_train_val = train_keys.intersection(val_keys)
    
    print("\n HASIL PEMBERSIHAN:")
    print("="*60)
    
    # Hapus Duplikat Train vs Test (Hapus yang di Test)
    if leak_train_test:
        print(f" Ditemukan {len(leak_train_test)} duplikat di TRAIN & TEST. Menghapus file di TEST...")
        for h in leak_train_test:
            file_to_remove = test_hashes[h]
            try:
                os.remove(file_to_remove)
                print(f"    Dihapus: {file_to_remove}")
            except Exception as e:
                print(f"    Gagal hapus {file_to_remove}: {e}")
    else:
        print(" Train vs Test: AMAN (Tidak ada duplikat).")

    # Hapus Duplikat Train vs Val (Hapus yang di Val)
    if leak_train_val:
        print(f" Ditemukan {len(leak_train_val)} duplikat di TRAIN & VAL. Menghapus file di VAL...")
        for h in leak_train_val:
            file_to_remove = val_hashes[h]
            try:
                os.remove(file_to_remove)
                print(f"    Dihapus: {file_to_remove}")
            except Exception as e:
                print(f"    Gagal hapus {file_to_remove}: {e}")
    else:
        print(" Train vs Val: AMAN (Tidak ada duplikat).")
        
    print("="*60)
    print(" Dataset sekarang sudah BERSIH dan VALID!")

# Jalankan Fungsi
find_and_remove_duplicates(TRAIN_DIR, VAL_DIR, TEST_DIR)


 SEDANG MEMERIKSA & MEMBERSIHKAN DUPLIKASI KONTEN...
   (Proses ini mungkin memakan waktu 1-2 menit...)
    Memproses TRAIN set...
    Memproses VAL set...
    Memproses TEST set...

 HASIL PEMBERSIHAN:
 Train vs Test: AMAN (Tidak ada duplikat).
 Train vs Val: AMAN (Tidak ada duplikat).
 Dataset sekarang sudah BERSIH dan VALID!


# Cek Data Leakage (Kebocoran Data Antar Split)

In [12]:

def check_data_leakage_by_filename(train_dir, val_dir, test_dir):
    """
    Mengecek kebocoran data berdasarkan NAMA FILE (Frame ID).
    Logika Baru:
    - Tidak masalah subjek (s0001) sama di Train/Test.
    - Yang HARAM adalah jika NAMA FILE (Frame) sama persis muncul di Train & Test.
    """
    print(" MEMERIKSA DATA LEAKAGE (BERDASARKAN DUPLIKASI NAMA FILE)...")
    print("="*60)

    # Helper untuk ambil semua nama file (tanpa path)
    def get_all_filenames(directory):
        filenames = set()
        count = 0
        for root, _, files in os.walk(directory):
            for file in files:
                if file.lower().endswith(('.jpg', '.png', '.jpeg')):
                    filenames.add(file)
                    count += 1
        return filenames, count

    # Ambil daftar nama file
    train_files, n_train = get_all_filenames(train_dir)
    val_files, n_val = get_all_filenames(val_dir)
    test_files, n_test = get_all_filenames(test_dir)

    print(f"   Jumlah Gambar di Train : {n_train}")
    print(f"   Jumlah Gambar di Val   : {n_val}")
    print(f"   Jumlah Gambar di Test  : {n_test}")

    # Cek Irisan (Apakah ada nama file yang sama persis?)
    leak_train_test = train_files.intersection(test_files)
    leak_train_val = train_files.intersection(val_files)
    leak_val_test = val_files.intersection(test_files)

    print("\n HASIL ANALISIS LEAKAGE:")
    
    # Laporan 1: Train vs Test
    if len(leak_train_test) > 0:
        print(f"    PERINGATAN: Ditemukan {len(leak_train_test)} file dengan nama SAMA di Train & Test!")
        print(f"      Ini berpotensi data leakage (Model diuji dengan gambar yang sama saat latihan).")
        print(f"      Contoh: {list(leak_train_test)[:3]}")
    else:
        print("    AMAN: Tidak ada file dengan nama yang sama antara Train & Test.")
        print("      (Validitas pengujian terjamin karena frame-nya unik).")

    # Laporan 2: Train vs Val
    if len(leak_train_val) > 0:
        print(f"    PERINGATAN: Ditemukan {len(leak_train_val)} file dengan nama SAMA di Train & Val!")
    else:
        print("    AMAN: Tidak ada file dengan nama yang sama antara Train & Val.")

    print("="*60)
    print(" KESIMPULAN AKHIR:")
    if len(leak_train_test) == 0 and len(leak_train_val) == 0:
        print("   Dataset VALID. Meskipun subjek (orang) sama muncul di berbagai split,")
        print("   tapi Frame/Gambar yang digunakan berbeda. Model akan belajar generalisasi pose mata,")
        print("   bukan menghafal identitas orang.")
    else:
        print("   Masih ada duplikasi nama file. Pastikan Anda sudah menjalankan kode penghapusan duplikat sebelumnya.")

# Jalankan Fungsi Revisi
check_data_leakage_by_filename(TRAIN_DIR, VAL_DIR, TEST_DIR)


 MEMERIKSA DATA LEAKAGE (BERDASARKAN DUPLIKASI NAMA FILE)...
   Jumlah Gambar di Train : 40949
   Jumlah Gambar di Val   : 18833
   Jumlah Gambar di Test  : 6743

 HASIL ANALISIS LEAKAGE:
    AMAN: Tidak ada file dengan nama yang sama antara Train & Test.
      (Validitas pengujian terjamin karena frame-nya unik).
    AMAN: Tidak ada file dengan nama yang sama antara Train & Val.
 KESIMPULAN AKHIR:
   Dataset VALID. Meskipun subjek (orang) sama muncul di berbagai split,
   tapi Frame/Gambar yang digunakan berbeda. Model akan belajar generalisasi pose mata,
   bukan menghafal identitas orang.
